# Primera prueba de visión en Colab

Ejecuta las celdas en orden y permanece en el cuaderno durante la prueba. Selecciona **Entorno de ejecución → Cambiar tipo de entorno → GPU**. No requiere cuenta de la aplicación, tokens, SSH ni exponer la API local.

Se comparan una **grabación** y el mismo archivo a **velocidad real**, usando el código del proyecto. La segunda modalidad es una simulación del procesamiento de directo; no mide una cámara remota ni latencia de red. Las capturas son evidencia para revisión, no anotaciones de verdad terreno.

Los pesos especializados y el checkpoint de 32 marcas de cancha todavía necesitan validación. COCO permite una primera prueba de personas y balón. Sin calibración no se producen posiciones en metros. [Colab no garantiza GPU ni duración de sesión](https://research.google.com/colaboratory/faq.html).

Cuaderno preparado localmente; la ejecución con GPU de Colab sigue pendiente.


## 1. Subir el código del proyecto

En la laptop, desde `vision`, ejecuta `..\.venv\Scripts\python -m evaluacion.preparar_colab`. Sube aquí el ZIP que indica ese comando. Incluye solo código y configuración del worker; excluye credenciales, videos, bases de datos y pesos.


In [ ]:
from google.colab import files
from pathlib import Path
import hashlib
import json
import subprocess
import sys
import zipfile

subidos = files.upload()
if len(subidos) != 1:
    raise ValueError("Selecciona solo analisisfutbol-colab.zip")
archivo_zip = next(iter(subidos))
raiz = Path('/content/analisisfutbol')
if raiz.exists():
    raise ValueError("El código ya se extrajo. Continúa con la siguiente celda o reinicia el entorno.")
with zipfile.ZipFile(archivo_zip) as z:
    if sum(i.file_size for i in z.infolist()) > 20_000_000:
        raise ValueError("El ZIP de código excede 20 MB")
    manifiesto = json.loads(z.read('manifiesto.json'))
    if set(z.namelist()) != {'manifiesto.json'} | {'vision/' + n for n in manifiesto}:
        raise ValueError("El paquete no coincide con su manifiesto")
    for nombre, digest in manifiesto.items():
        destino = (raiz / 'vision' / nombre).resolve()
        if not destino.is_relative_to(raiz.resolve()):
            raise ValueError("Ruta no permitida en el paquete")
        contenido = z.read('vision/' + nombre)
        if hashlib.sha256(contenido).hexdigest() != digest:
            raise ValueError("El archivo no coincide con su hash: " + nombre)
        destino.parent.mkdir(parents=True, exist_ok=True)
        destino.write_bytes(contenido)
(raiz / 'manifiesto.json').write_text(json.dumps(manifiesto, indent=2))
subidos.clear()
print('Código verificado:', len(manifiesto), 'archivos')


## 2. Preparar el entorno y comprobar la GPU

La instalación utiliza los requisitos del worker. El informe final conserva las versiones exactas y la información de GPU. No se instala nada en la laptop.


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(raiz / 'vision/requirements.txt')], check=True)
sys.path.insert(0, str(raiz / 'vision'))
import torch
if not torch.cuda.is_available():
    raise RuntimeError('No hay GPU disponible. Cambia el tipo de entorno y vuelve a ejecutar el cuaderno.')
subprocess.run(['nvidia-smi'], check=True)
print('GPU:', torch.cuda.get_device_name(0), '| Torch:', torch.__version__)


## 3. Subir el video y revisar el primer cuadro

Sube el archivo Bayern–PSG o un clip autorizado. Para la primera prueba se procesarán solo 30 segundos. Puedes descargar `primer-cuadro.png` desde Archivos de Colab y usarlo en **Calibrar cámara** de la aplicación local, sin cambiar su resolución.


In [ ]:
import cv2
from IPython.display import Image, display
subidos = files.upload()
if len(subidos) != 1:
    raise ValueError('Selecciona un solo video')
video = Path(next(iter(subidos))).resolve()
subidos.clear()
cap = cv2.VideoCapture(str(video))
try:
    ok, primer_cuadro = cap.read()
    if not ok:
        raise ValueError('No se pudo decodificar el video')
    cv2.imwrite('/content/primer-cuadro.png', primer_cuadro)
finally:
    cap.release()
display(Image(filename='/content/primer-cuadro.png', width=1000))
print('Video:', video.name, '| Resolución:', primer_cuadro.shape[1], 'x', primer_cuadro.shape[0])


## 4. Calibración opcional

Para probar solo detección y seguimiento, conserva `USAR_CALIBRACION = False`.

Para probar el pipeline en metros: calibra **ese primer cuadro** en la aplicación, guarda la calibración válida y pulsa **Descargar calibración**. Cambia la variable a `True` y sube ese JSON. No uses una matriz identidad ni puntos de un encuadre distinto. La referencia puede perderse durante el paneo; esto debe aparecer en los resultados. Los lentes con distorsión requieren un video rectificado y calibración correspondiente.


In [ ]:
USAR_CALIBRACION = False
calibracion = None
if USAR_CALIBRACION:
    subidos = files.upload()
    if len(subidos) != 1:
        raise ValueError('Selecciona solo el JSON de calibración')
    calibracion = json.loads(next(iter(subidos.values())))
    subidos.clear()
    if not calibracion.get('valida') or not calibracion.get('matriz_h'):
        raise ValueError('La calibración no es válida')
print('Pipeline métrico:', calibracion is not None)


## 5. Configuración y pesos

COCO descarga `yolo11n.pt` desde el mecanismo oficial de Ultralytics. No distingue árbitros como clase propia. Para un modelo especializado, cambia `SUBIR_PESOS`, selecciona el mapa correcto y sube solo un checkpoint de procedencia confiable. El código verifica las etiquetas, pero su precisión requiere anotaciones reales. Un modelo de pose humana no reemplaza al modelo de marcas de cancha.


In [ ]:
PESOS = 'yolo11n.pt'
CLASES = 'coco'          # coco, futbol (4 clases) o propio (3 clases)
SUBIR_PESOS = False
TAMANO_IMG = 1280
SEGUNDOS = 30
STRIDE = 1
COLOR_PROPIO = '#C8102E'
COLOR_RIVAL = '#17202A'
ES_LOCAL = True
if SUBIR_PESOS:
    subidos = files.upload()
    if len(subidos) != 1:
        raise ValueError('Selecciona un checkpoint')
    PESOS = str(Path(next(iter(subidos))).resolve())
    subidos.clear()
from deteccion.modelo import CLASES_COCO, CLASES_FUTBOL, CLASES_PROPIO, ConfigModelo, ModeloDeteccion
from evaluacion.ensayo_video import ensayar
from datetime import datetime, timezone
salidas = Path('/content/resultados-' + datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S'))
salidas.mkdir(exist_ok=False)
(salidas / 'manifiesto-codigo.json').write_text((raiz / 'manifiesto.json').read_text())
(salidas / 'entorno.txt').write_bytes(subprocess.check_output([sys.executable, '-m', 'pip', 'freeze']))
(salidas / 'gpu.txt').write_bytes(subprocess.check_output(['nvidia-smi']))
def nuevo_modelo():
    mapa = {'coco': CLASES_COCO, 'futbol': CLASES_FUTBOL, 'propio': CLASES_PROPIO}[CLASES]
    m = ModeloDeteccion(ConfigModelo(ruta_pesos=PESOS, clases=dict(mapa), dispositivo='cuda:0', tamano_img=TAMANO_IMG))
    m.cargar()
    return m


## 6. Procesar grabación

Se conservan los tiempos del archivo, las detecciones y sus IDs. Las muestras métricas se publican como máximo a 5 Hz cuando hay calibración. Cada ensayo utiliza un modelo nuevo para no reutilizar IDs o estado de la ejecución anterior. Si aparece falta de memoria, reduce `TAMANO_IMG` a 960 y crea otro ensayo; conserva ambos informes para comparar.

Al terminar se muestra el dispositivo real de los pesos (`cuda:0`), el nombre de la GPU y la memoria maxima utilizada. Estos datos tambien se guardan en el resumen. Una GPU al 0% despues de terminar el procesamiento es normal.


In [ ]:
from uuid import uuid4
carpeta_grabacion = salidas / ('grabacion-' + uuid4().hex[:8])
modelo = nuevo_modelo()
torch.cuda.reset_peak_memory_stats()
res_grabacion = ensayar(modelo, video, carpeta_grabacion, segundos=SEGUNDOS, stride=STRIDE,
    calibracion=calibracion, color_propio=COLOR_PROPIO, color_rival=COLOR_RIVAL, es_local=ES_LOCAL)
torch.cuda.synchronize()
dispositivo_real = res_grabacion.get('dispositivo')
print('Dispositivo real del modelo:', dispositivo_real)
if not dispositivo_real or not dispositivo_real.startswith('cuda'):
    raise RuntimeError('El modelo no se ejecuto en GPU')
print('Memoria GPU maxima usada (MiB):', round(torch.cuda.max_memory_allocated() / 1024**2, 1))
res_grabacion['dispositivo_real_modelo'] = str(dispositivo_real)
res_grabacion['gpu_nombre'] = torch.cuda.get_device_name(0)
res_grabacion['gpu_memoria_max_mib'] = round(torch.cuda.max_memory_allocated() / 1024**2, 1)
(carpeta_grabacion / 'resumen.json').write_text(json.dumps(res_grabacion, ensure_ascii=False, indent=2), encoding='utf-8')
del modelo
torch.cuda.empty_cache()
print(json.dumps(res_grabacion, ensure_ascii=False, indent=2))


## 7. Simular la fuente de directo

Reproduce el mismo archivo a velocidad real y conserva solo el cuadro más reciente cuando la inferencia se retrasa. Esta prueba usa el lector de directo del proyecto. No abre servidores ni conecta Colab por SSH. No certifica transporte de cámara, WebSocket, estabilidad durante 90 minutos ni latencia extremo a extremo.


In [ ]:
from uuid import uuid4
carpeta_directo = salidas / ('directo-simulado-' + uuid4().hex[:8])
modelo = nuevo_modelo()
torch.cuda.reset_peak_memory_stats()
res_directo = ensayar(modelo, video, carpeta_directo, segundos=SEGUNDOS, simular_directo=True,
    calibracion=calibracion, color_propio=COLOR_PROPIO, color_rival=COLOR_RIVAL, es_local=ES_LOCAL)
torch.cuda.synchronize()
dispositivo_real = res_directo.get('dispositivo')
print('Dispositivo real del modelo:', dispositivo_real)
if not dispositivo_real or not dispositivo_real.startswith('cuda'):
    raise RuntimeError('El modelo no se ejecuto en GPU')
print('Memoria GPU maxima usada (MiB):', round(torch.cuda.max_memory_allocated() / 1024**2, 1))
res_directo['dispositivo_real_modelo'] = str(dispositivo_real)
res_directo['gpu_nombre'] = torch.cuda.get_device_name(0)
res_directo['gpu_memoria_max_mib'] = round(torch.cuda.max_memory_allocated() / 1024**2, 1)
(carpeta_directo / 'resumen.json').write_text(json.dumps(res_directo, ensure_ascii=False, indent=2), encoding='utf-8')
del modelo
torch.cuda.empty_cache()
print(json.dumps(res_directo, ensure_ascii=False, indent=2))


## 8. Revisar y descargar evidencia

Revisa especialmente balón pequeño, árbitro, personas fuera del terreno y paneos. `detecciones.jsonl` contiene cajas en píxeles. `muestras.json` contiene posiciones métricas si se proporcionó calibración; queda vacío en caso contrario. No se importan automáticamente a la base de datos de la aplicación.

Los contadores de detección y la velocidad no son medidas de precisión. Para mAP/IDF1 se necesitan anotaciones independientes y el evaluador `evaluacion.regresion` del proyecto.


In [ ]:
for carpeta in sorted(salidas.iterdir()):
    if carpeta.is_dir():
        print(carpeta.name)
        for foto in sorted(carpeta.glob('cuadro-*.jpg'))[:3]:
            display(Image(filename=str(foto), width=1000))
import shutil
paquete_resultados = shutil.make_archive(str(salidas), 'zip', salidas)
files.download(paquete_resultados)
print('Descarga completa antes de desconectar el entorno. Conserva el ZIP para revisar los resultados.')
